# 🌲 Binary Trees — Runnable Notebook

Companion to [`README.md`](README.md) and
[`02_binary_tree_lesson.html`](02_binary_tree_lesson.html).

A **binary tree** gives every node **at most two ordered children** (`left`, `right`).

## 1. The node and a sample tree

In [ ]:
class TreeNode:
    """A binary-tree node: a value and up to two children (left, right)."""
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

# Build:
#          1
#         / \
#        2   3
#       / \   \
#      4   5    6
root = TreeNode(1,
    TreeNode(2, TreeNode(4), TreeNode(5)),
    TreeNode(3, None, TreeNode(6)))
print("root value:", root.val)

## 2. Size and height — the classic recursion
**Solve `left`, solve `right`, combine.**

In [ ]:
def size(node):
    """Total nodes = 1 + left subtree + right subtree."""
    if not node:
        return 0
    return 1 + size(node.left) + size(node.right)

def height(node):
    """Longest path down in EDGES. Empty = -1 so a leaf = 0."""
    if not node:
        return -1
    return 1 + max(height(node.left), height(node.right))

print("size  :", size(root))
print("height:", height(root))
assert size(root) == 6 and height(root) == 2

## 3. Invert (mirror) the tree
Swap every node's left and right. A neat fact: mirroring **reverses** the in-order traversal.

In [ ]:
def inorder(node, out=None):
    """Left, Node, Right."""
    out = [] if out is None else out
    if node:
        inorder(node.left, out); out.append(node.val); inorder(node.right, out)
    return out

def invert(node):
    """Mirror the tree: swap left/right everywhere (LeetCode 226)."""
    if not node:
        return None
    node.left, node.right = invert(node.right), invert(node.left)   # swap, then recurse
    return node

before = inorder(root)
invert(root)
after = inorder(root)
print("in-order before:", before)
print("in-order after :", after)
assert after == before[::-1]        # mirroring reverses in-order
invert(root)                        # invert again to restore the original tree

## 4. Diameter — "return one value, track another"
The longest path between any two nodes. A single DFS **returns each node's height**, while a side variable
**records the best `left + right`** seen anywhere.

In [ ]:
def diameter(root):
    """Longest path (in edges) between ANY two nodes — may not pass through the root."""
    best = 0
    def depth(node):
        nonlocal best
        if not node:
            return 0
        L = depth(node.left)
        R = depth(node.right)
        best = max(best, L + R)      # a path THROUGH this node uses L + R edges
        return 1 + max(L, R)         # but we report HEIGHT up to the parent
    depth(root)
    return best

print("diameter:", diameter(root))
# longest path here is 4-2-1-3-6 = 4 edges
assert diameter(root) == 4

## 5. Array packing (how a heap is stored)
Lay a tree out level by level into an array; then links are pure arithmetic:
`left = 2i+1`, `right = 2i+2`, `parent = (i-1)//2`.

In [ ]:
def from_array(arr):
    """Build a binary tree from a level-order array (None = missing child)."""
    if not arr:
        return None
    nodes = [TreeNode(v) if v is not None else None for v in arr]
    for i, node in enumerate(nodes):
        if node is None:
            continue
        li, ri = 2*i + 1, 2*i + 2     # the index formula
        if li < len(arr): node.left  = nodes[li]
        if ri < len(arr): node.right = nodes[ri]
    return nodes[0]

def to_array(root, n):
    """Pack a COMPLETE tree back into an array using the same index rule."""
    arr = [None] * n
    def fill(node, i):
        if node is None or i >= n:
            return
        arr[i] = node.val
        fill(node.left,  2*i + 1)
        fill(node.right, 2*i + 2)
    fill(root, 0)
    return arr

packed = [1, 2, 3, 4, 5, 6, 7]        # a perfect tree
t = from_array(packed)
print("rebuilt in-order :", inorder(t))
print("re-packed array  :", to_array(t, 7))
assert to_array(t, 7) == packed
i = 1
print(f"index {i}: children at {2*i+1} and {2*i+2}, parent at {(i-1)//2}")

## ✅ Recap
- At most **two ordered children** (`left`, `right`).
- Recursion shape: **solve left, solve right, combine**.
- **Diameter / balance / max-path** use "return height up, track the answer in a side variable".
- **Complete** trees pack into an array with `left=2i+1`, `right=2i+2` — no pointers (this is a heap).

Next: [`03_Binary_Search_Tree`](../03_Binary_Search_Tree/README.md).